# 03 - Bronze: trazendo os dados para dentro do Snowflake

**Objetivo:** pegar os arquivos CSV da Loja Paulista (que estão no GitHub) e carregá-los em tabelas, **sem mexer em nada**.

**Analogia:** é o momento em que os ingredientes chegam do mercado e vão para a despensa. Ainda tem terra na batata, embalagem amassada, um item repetido. Tudo bem: na camada Bronze guardamos **exatamente** o que chegou.

```
 GitHub (CSVs) --COPY FILES--> Stage ARQUIVOS --COPY INTO--> Tabelas BRONZE
```

**Tempo estimado:** 25 minutos

## Passo 0 - Contexto

In [ ]:
%%sql -r dataframe_1
-- Contexto da sessao: quem sou eu, qual motor e qual gaveta vou usar.
-- Rode esta celula SEMPRE que abrir o notebook.
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE ESPM_WH;
USE DATABASE ESPM_HOL;
USE SCHEMA BRONZE;

## Passo 1 - Uma "área de pouso" para os arquivos (stage)

**O que é:** um *stage* é uma pasta dentro do Snowflake onde arquivos ficam antes de virarem tabela.

No setup criamos o `HOL_REPO`, que é um espelho do GitHub. Agora copiamos os CSVs do espelho para um stage nosso, chamado `ARQUIVOS`.

In [ ]:
%%sql -r dataframe_2
-- Cria a pasta interna (stage) onde os arquivos vao pousar
CREATE STAGE IF NOT EXISTS ESPM_HOL.BRONZE.ARQUIVOS
  COMMENT = 'Area de pouso dos CSVs da Loja Paulista';

-- Garante que temos a versao mais nova do GitHub
ALTER GIT REPOSITORY ESPM_HOL.PUBLIC.HOL_REPO FETCH;

-- Copia os arquivos do espelho do GitHub para o nosso stage
COPY FILES INTO @ESPM_HOL.BRONZE.ARQUIVOS/
  FROM @ESPM_HOL.PUBLIC.HOL_REPO/branches/main/data/;

-- Lista o que pousou: devem aparecer 9 arquivos
LIST @ESPM_HOL.BRONZE.ARQUIVOS;

**O que observar:** a coluna `name` mostra cada arquivo e `size`, o tamanho em bytes. Nenhum dado foi lido ainda; só movemos arquivos.

## Passo 2 - Ensinar o Snowflake a ler o CSV (file format)

**O que é:** o *file format* é a "receita de leitura": separador, se a primeira linha é cabeçalho, como tratar campo vazio.

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE FILE FORMAT ESPM_HOL.BRONZE.FF_CSV
  TYPE = CSV
  PARSE_HEADER = TRUE                 -- a 1a linha tem os nomes das colunas
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'  -- textos podem vir entre aspas
  EMPTY_FIELD_AS_NULL = TRUE          -- campo vazio vira NULL (ausencia de valor)
  TRIM_SPACE = FALSE;                 -- NAO limpamos espacos: Bronze guarda o dado cru

## Passo 3 - Criar as tabelas Bronze

Todas as colunas são texto (`VARCHAR`). De propósito: se um arquivo vier com uma data esquisita, o carregamento **não quebra**. A conversão de tipos é trabalho da camada Silver.

Duas colunas extras de **auditoria** contam de onde veio cada linha e quando ela chegou: `_arquivo` e `_carregado_em`.

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE TABLE BRONZE.LOJAS (loja_id VARCHAR, nome_loja VARCHAR, cidade VARCHAR, uf VARCHAR,
  tipo_loja VARCHAR, data_abertura VARCHAR, _arquivo VARCHAR, _carregado_em TIMESTAMP_LTZ);

CREATE OR REPLACE TABLE BRONZE.PRODUTOS (produto_id VARCHAR, nome_produto VARCHAR, categoria VARCHAR,
  preco_lista VARCHAR, _arquivo VARCHAR, _carregado_em TIMESTAMP_LTZ);

CREATE OR REPLACE TABLE BRONZE.CANAIS_CAMPANHA (canal_id VARCHAR, nome_canal VARCHAR, tipo_canal VARCHAR,
  custo_mensal VARCHAR, _arquivo VARCHAR, _carregado_em TIMESTAMP_LTZ);

CREATE OR REPLACE TABLE BRONZE.CLIENTES (cliente_id VARCHAR, nome VARCHAR, email VARCHAR, cidade VARCHAR,
  uf VARCHAR, data_cadastro VARCHAR, canal_aquisicao_id VARCHAR, atualizado_em VARCHAR,
  _arquivo VARCHAR, _carregado_em TIMESTAMP_LTZ);

CREATE OR REPLACE TABLE BRONZE.PEDIDOS (pedido_id VARCHAR, cliente_id VARCHAR, loja_id VARCHAR,
  canal_venda VARCHAR, data_pedido VARCHAR, status VARCHAR, cupom VARCHAR, valor_frete VARCHAR,
  valor_total VARCHAR, _arquivo VARCHAR, _carregado_em TIMESTAMP_LTZ);

CREATE OR REPLACE TABLE BRONZE.ITENS_PEDIDO (item_id VARCHAR, pedido_id VARCHAR, produto_id VARCHAR,
  quantidade VARCHAR, preco_unitario VARCHAR, desconto VARCHAR, _arquivo VARCHAR, _carregado_em TIMESTAMP_LTZ);

CREATE OR REPLACE TABLE BRONZE.CAMPANHAS_DISPAROS (disparo_id VARCHAR, canal_id VARCHAR, cliente_id VARCHAR,
  data_disparo VARCHAR, abriu VARCHAR, clicou VARCHAR, pedido_convertido_id VARCHAR,
  _arquivo VARCHAR, _carregado_em TIMESTAMP_LTZ);

CREATE OR REPLACE TABLE BRONZE.EVENTOS_APP (evento_id VARCHAR, cliente_id VARCHAR, ts_evento VARCHAR,
  tipo_evento VARCHAR, tela VARCHAR, plataforma VARCHAR, _arquivo VARCHAR, _carregado_em TIMESTAMP_LTZ);

CREATE OR REPLACE TABLE BRONZE.REVIEWS (review_id VARCHAR, cliente_id VARCHAR, produto_id VARCHAR,
  nota VARCHAR, data_review VARCHAR, comentario VARCHAR, _arquivo VARCHAR, _carregado_em TIMESTAMP_LTZ);

## Passo 4 - Carregar (COPY INTO)

**O que é:** o `COPY INTO` lê o arquivo do stage e grava as linhas na tabela. `MATCH_BY_COLUMN_NAME` liga cada coluna do CSV à coluna de mesmo nome na tabela. `INCLUDE_METADATA` preenche as colunas de auditoria.

O Snowflake **lembra quais arquivos já carregou**: se você rodar de novo, ele não duplica.

In [ ]:
%%sql -r dataframe_5
COPY INTO BRONZE.LOJAS FROM @BRONZE.ARQUIVOS FILES = ('lojas.csv')
  FILE_FORMAT = (FORMAT_NAME = 'ESPM_HOL.BRONZE.FF_CSV') MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
  INCLUDE_METADATA = (_arquivo = METADATA$FILENAME, _carregado_em = METADATA$START_SCAN_TIME);

COPY INTO BRONZE.PRODUTOS FROM @BRONZE.ARQUIVOS FILES = ('produtos.csv')
  FILE_FORMAT = (FORMAT_NAME = 'ESPM_HOL.BRONZE.FF_CSV') MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
  INCLUDE_METADATA = (_arquivo = METADATA$FILENAME, _carregado_em = METADATA$START_SCAN_TIME);

COPY INTO BRONZE.CANAIS_CAMPANHA FROM @BRONZE.ARQUIVOS FILES = ('canais_campanha.csv')
  FILE_FORMAT = (FORMAT_NAME = 'ESPM_HOL.BRONZE.FF_CSV') MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
  INCLUDE_METADATA = (_arquivo = METADATA$FILENAME, _carregado_em = METADATA$START_SCAN_TIME);

COPY INTO BRONZE.CLIENTES FROM @BRONZE.ARQUIVOS FILES = ('clientes.csv')
  FILE_FORMAT = (FORMAT_NAME = 'ESPM_HOL.BRONZE.FF_CSV') MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
  INCLUDE_METADATA = (_arquivo = METADATA$FILENAME, _carregado_em = METADATA$START_SCAN_TIME);

COPY INTO BRONZE.PEDIDOS FROM @BRONZE.ARQUIVOS FILES = ('pedidos.csv')
  FILE_FORMAT = (FORMAT_NAME = 'ESPM_HOL.BRONZE.FF_CSV') MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
  INCLUDE_METADATA = (_arquivo = METADATA$FILENAME, _carregado_em = METADATA$START_SCAN_TIME);

COPY INTO BRONZE.ITENS_PEDIDO FROM @BRONZE.ARQUIVOS FILES = ('itens_pedido.csv')
  FILE_FORMAT = (FORMAT_NAME = 'ESPM_HOL.BRONZE.FF_CSV') MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
  INCLUDE_METADATA = (_arquivo = METADATA$FILENAME, _carregado_em = METADATA$START_SCAN_TIME);

COPY INTO BRONZE.CAMPANHAS_DISPAROS FROM @BRONZE.ARQUIVOS FILES = ('campanhas_disparos.csv')
  FILE_FORMAT = (FORMAT_NAME = 'ESPM_HOL.BRONZE.FF_CSV') MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
  INCLUDE_METADATA = (_arquivo = METADATA$FILENAME, _carregado_em = METADATA$START_SCAN_TIME);

COPY INTO BRONZE.EVENTOS_APP FROM @BRONZE.ARQUIVOS FILES = ('eventos_app.csv')
  FILE_FORMAT = (FORMAT_NAME = 'ESPM_HOL.BRONZE.FF_CSV') MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
  INCLUDE_METADATA = (_arquivo = METADATA$FILENAME, _carregado_em = METADATA$START_SCAN_TIME);

COPY INTO BRONZE.REVIEWS FROM @BRONZE.ARQUIVOS FILES = ('reviews.csv')
  FILE_FORMAT = (FORMAT_NAME = 'ESPM_HOL.BRONZE.FF_CSV') MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
  INCLUDE_METADATA = (_arquivo = METADATA$FILENAME, _carregado_em = METADATA$START_SCAN_TIME);

### Checkpoint: quantas linhas chegaram?

In [ ]:
%%sql -r dataframe_6
-- Esperado: canais 10 | clientes 300 | disparos 200 | eventos 400 | itens 600
--           lojas 12 | pedidos 300 | produtos 80 | reviews 40
SELECT 'campanhas_disparos' AS tabela, COUNT(*) AS linhas FROM BRONZE.CAMPANHAS_DISPAROS UNION ALL
SELECT 'canais_campanha', COUNT(*) FROM BRONZE.CANAIS_CAMPANHA UNION ALL
SELECT 'clientes',        COUNT(*) FROM BRONZE.CLIENTES        UNION ALL
SELECT 'eventos_app',     COUNT(*) FROM BRONZE.EVENTOS_APP     UNION ALL
SELECT 'itens_pedido',    COUNT(*) FROM BRONZE.ITENS_PEDIDO    UNION ALL
SELECT 'lojas',           COUNT(*) FROM BRONZE.LOJAS           UNION ALL
SELECT 'pedidos',         COUNT(*) FROM BRONZE.PEDIDOS         UNION ALL
SELECT 'produtos',        COUNT(*) FROM BRONZE.PRODUTOS        UNION ALL
SELECT 'reviews',         COUNT(*) FROM BRONZE.REVIEWS
ORDER BY tabela;

## Passo 5 - SQL em 5 minutos: fazendo perguntas aos dados

SQL é a língua para conversar com tabelas. Toda pergunta de negócio vira uma consulta:

| Pergunta | Pedaço do SQL |
|---|---|
| De onde? | `FROM pedidos` |
| Quais colunas? | `SELECT canal_venda, valor_total` |
| Só alguns? | `WHERE status = 'ENTREGUE'` |
| Agrupar e contar? | `GROUP BY canal_venda` + `COUNT(*)` |

In [ ]:
%%sql -r dataframe_7
-- Uma espiada nos 10 primeiros pedidos
SELECT * FROM BRONZE.PEDIDOS LIMIT 10;

**O que observar:** a coluna `data_pedido` tem **três formatos diferentes** (`2026-03-22`, `28/08/2026`, `2026-05-01 14:32:00`). Guarde isso: vamos consertar na Silver.

### EXERCICIO 1 - Quantos pedidos foram feitos pelo APP?

Complete os `???` na célula abaixo.

**Dica:** a coluna que guarda por onde a venda aconteceu se chama `canal_venda`.

**Travou?** Abra o Cortex Code e peça:
> Escreva um SQL que conte os pedidos da tabela ESPM_HOL.BRONZE.PEDIDOS cujo canal de venda é APP

In [ ]:
%%sql -r dataframe_8
SELECT COUNT(*) AS pedidos_app
FROM BRONZE.PEDIDOS
WHERE ??? = 'APP';

### EXERCICIO 2 - Quantos pedidos existem em cada status (ENTREGUE, CANCELADO...)?

Complete os `???` na célula abaixo.

**Dica:** você agrupa pela mesma coluna que aparece no SELECT ao lado do COUNT.

**Travou?** Abra o Cortex Code e peça:
> Mostre quantos pedidos existem por status na tabela ESPM_HOL.BRONZE.PEDIDOS, do maior para o menor

In [ ]:
%%sql -r dataframe_9
SELECT status, COUNT(*) AS pedidos
FROM BRONZE.PEDIDOS
GROUP BY ???
ORDER BY pedidos DESC;

### EXERCICIO 3 - Encontre clientes que aparecem mais de uma vez (duplicados)

Complete os `???` na célula abaixo.

**Dica:** `HAVING` é um WHERE que filtra depois de agrupar. Duplicado = aparece mais de 1 vez.

**Travou?** Abra o Cortex Code e peça:
> Liste os cliente_id que aparecem mais de uma vez em ESPM_HOL.BRONZE.CLIENTES

In [ ]:
%%sql -r dataframe_10
SELECT cliente_id, COUNT(*) AS vezes
FROM BRONZE.CLIENTES
GROUP BY cliente_id
HAVING COUNT(*) > ???
ORDER BY cliente_id;

## Resumo

- Os arquivos saíram do GitHub, pousaram num **stage** e viraram **tabelas Bronze**.
- Nada foi limpo: há **datas misturadas**, **e-mails com maiúsculas e espaços**, **15 clientes duplicados** e **10 pedidos sem valor**.

**Próximo:** `04_silver_dynamic_tables` - vamos limpar tudo isso automaticamente.